In [50]:
import polars as pl
from haystack import Document
from haystack.components.evaluators import DocumentNDCGEvaluator, DocumentMAPEvaluator, DocumentRecallEvaluator
from haystack.components.evaluators.document_recall import RecallMode

from app.word2vec.pipeline import create as create_w2v
from app.sbert.pipeline import create as create_sbert
from app.llm.pipeline import create as create_llm

In [48]:
datasets = pl.read_csv("./data/fyp.csv") \
    .filter(pl.col("abstract").is_not_null()) \
    .unique("title")
datasets

title,abstract,webpage_url,document_url
str,str,str,str
"""A Study on the Efficiency of t…","""Building Information Modelling…","""https://eprints.tarc.edu.my/23…","""https://eprints.tarc.edu.my/23…"
"""Industrialized Building System…","""IBS (Industrialized Building S…","""https://eprints.tarc.edu.my/91…","""https://eprints.tarc.edu.my/91…"
"""Design and Fabrication of a No…","""This project was initially con…","""https://eprints.tarc.edu.my/96…","""https://eprints.tarc.edu.my/96…"
"""Impact of Jet Fuel Price on th…","""This study attempts to investi…","""https://eprints.tarc.edu.my/18…","""https://eprints.tarc.edu.my/18…"
"""Standardize Troubleshooting Sy…","""In continuous inkjet technolog…","""https://eprints.tarc.edu.my/98…","""https://eprints.tarc.edu.my/98…"
…,…,…,…
"""In Vitro Antagonism Activity o…","""Elaeis guineensis or commonly …","""https://eprints.tarc.edu.my/22…","""https://eprints.tarc.edu.my/22…"
"""Peanut Butter Matrix as a Pote…","""Probiotics-rich foods are incr…","""https://eprints.tarc.edu.my/25…","""https://eprints.tarc.edu.my/25…"
"""Tomato Grading System by Image…","""Nowadays, consumer demand more…","""https://eprints.tarc.edu.my/14…","""https://eprints.tarc.edu.my/14…"


In [49]:
documents = [Document(id=row["title"], content=row["abstract"]) for row in datasets.iter_rows(named=True)]
documents

[Document(id=A Study on the Efficiency of the Implementation of Building Information Modelling (BIM) in the Construction Industry, content: 'Building Information Modelling (BIM) is a technology that is currently gaining momentum within the c...'),
 Document(id=Industrialized Building System (IBS) in Malaysia Construction Industry, content: 'IBS (Industrialized Building System) is defined as a construction system which all the components ma...'),
 Document(id=Design and Fabrication of a Non-Electrical Driven Positive Displacement Pump, content: 'This project was initially conceived to be a solar-powered water pump with the aim of transporting w...'),
 Document(id=Impact of Jet Fuel Price on the Returns of Airlines Stock Price in ASEAN-5, content: 'This study attempts to investigate the impact of the jet fuel price on the returns on the airline st...'),
 Document(id=Standardize Troubleshooting System For An Ink Jet Printer Servicing Company, content: 'In continuous inkjet technology, a h

In [19]:
pipeline = (create_w2v(documents, force_embeddings=False)
            # .with_hyde()
            # .with_bm25()
            # .with_evaluator()
            .make())

50it [00:00, 1053.68it/s]


In [21]:
query = "Design of automated and robotic systems"
top_k = 10
results = pipeline.run(data={
    "query": {"query": query, "top_k": top_k}
}, include_outputs_from={"result", "evaluator"})
results

1it [00:00, 2100.30it/s]


{'result': {'documents': [Document(id=Design and Build Delivery Robot, content: 'Design and build a delivery robot that uses GPS technology to navigate effectively and autonomously ...', score: 0.6697127289171143),
   Document(id=Robot Control Interface with Video Feedback for Wireless Camera, content: 'The title of this project is "Robot Control Interface with Video Feed Back for Wireless Camera". Thi...', score: 0.5733347975141693),
   Document(id=Design of an Automated Powder Spray System for Cable Trunk, content: 'Throughout this Final Year Project, the recessed area of the cable trunk is needed to be sprayed by ...', score: 0.5383138688097516),
   Document(id=Design and Development of Low-Cost Wind Powered Motorcycle Mobile Charger, content: 'Smart devices have become an essential that now acts as an extension of oneself that not only connec...', score: 0.5154608565270495),
   Document(id=Disturbance Force Compensation Using Robust Controller in Linear Drive Positioning System, co

In [33]:
ground_truth = datasets.filter(pl.col("title").is_in((
    "Robot Control Interface with Video Feedback for Wireless Camera",
    "Design and Build Delivery Robot",
    "Arduino Based Dual-Axis Solar Tracking System",
    "Identification in the Presence of Speed Bumps through Few-Shot Learning",
    "Robot Control Interface with Video Feedback for Wireless Camera",
    "Single-Sample Face Recognition for Attendance Record",
    "Cable Fault Recognition Using Image Analysis Of Phase Resolved Partial Discharge Pattern",
    "Banana Leaf Disease Detection Using Image Processing Methods",
    "Design of an Automated Powder Spray System for Cable Trunk",
    "Design and Develop a Cost-Effective Hydroponics Autodoser System",
)))
ground_truth

title,abstract,webpage_url,document_url
str,str,str,str
"""Single-Sample Face Recognition…","""Face recognition system is wid…","""https://eprints.tarc.edu.my/18…","""https://eprints.tarc.edu.my/18…"
"""Design of an Automated Powder …","""Throughout this Final Year Pro…","""https://eprints.tarc.edu.my/35…","""https://eprints.tarc.edu.my/35…"
"""Design and Develop a Cost-Effe…","""Hydroponic systems offer susta…","""https://eprints.tarc.edu.my/31…","""https://eprints.tarc.edu.my/31…"
"""Arduino Based Dual-Axis Solar …","""The project that this paper is…","""https://eprints.tarc.edu.my/33…","""https://eprints.tarc.edu.my/33…"
"""Design and Build Delivery Robo…","""Design and build a delivery ro…","""https://eprints.tarc.edu.my/31…","""https://eprints.tarc.edu.my/31…"
"""Identification in the Presence…","""Speed bump is one of the major…","""https://eprints.tarc.edu.my/22…","""https://eprints.tarc.edu.my/22…"
"""Banana Leaf Disease Detection …","""This paper presents a detectio…","""https://eprints.tarc.edu.my/14…","""https://eprints.tarc.edu.my/14…"
"""Cable Fault Recognition Using …","""Partial Discharge (PD) source …","""https://eprints.tarc.edu.my/23…","""https://eprints.tarc.edu.my/23…"
"""Robot Control Interface with V…","""The title of this project is ""…","""https://eprints.tarc.edu.my/92…","""https://eprints.tarc.edu.my/92…"


In [35]:
ground_truth_documents = [Document(id=row["title"], content=row["abstract"]) for row in ground_truth.iter_rows(named=True)]
ground_truth_documents

[Document(id=Single-Sample Face Recognition for Attendance Record, content: 'Face recognition system is widely used in the world; it had made the human live become more convenie...'),
 Document(id=Design of an Automated Powder Spray System for Cable Trunk, content: 'Throughout this Final Year Project, the recessed area of the cable trunk is needed to be sprayed by ...'),
 Document(id=Design and Develop a Cost-Effective Hydroponics Autodoser System, content: 'Hydroponic systems offer sustainable benefits, but manual nutrient dosing methods pose challenges in...'),
 Document(id=Arduino Based Dual-Axis Solar Tracking System, content: 'The project that this paper is presenting is about preparing, building, and testing a powerful solar...'),
 Document(id=Design and Build Delivery Robot, content: 'Design and build a delivery robot that uses GPS technology to navigate effectively and autonomously ...'),
 Document(id=Identification in the Presence of Speed Bumps through Few-Shot Learning, cont

In [24]:
precision = 5 / 10
precision

0.5

In [25]:
recall = 5 / 9
recall

0.5555555555555556

In [31]:
fallout = (50 - 5) / (50 - 9)
fallout

1.0975609756097562

In [27]:
f = (2 * precision * recall) / (precision + recall)
f

0.5263157894736842

In [29]:
f_b = lambda b: (1 + b ** 2) * (precision * recall) / (b ** 2 * precision + recall)
f_b(2)

0.5434782608695652

In [47]:
recall = DocumentRecallEvaluator(mode=RecallMode.MULTI_HIT)
recall_result = recall.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
recall_result

{'score': 0.5555555555555556, 'individual_scores': [0.5555555555555556]}

In [40]:
ndcg = DocumentNDCGEvaluator()
ndcg_result = ndcg.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
ndcg_result

{'score': 0.649969831884453, 'individual_scores': [0.649969831884453]}

In [42]:
map = DocumentMAPEvaluator()
map_result = map.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
map_result

{'score': 0.8253968253968254, 'individual_scores': [0.8253968253968254]}